# 풀스택 GPT: #6.0~6.10

## Tasks:

- [x] 체인에 다음 질문을 합니다:
    > - [x] Aaronson 은 유죄인가요?
    > - [x] 그가 테이블에 어떤 메시지를 썼나요?
    > - [x] Julia 는 누구인가요?
- [x] Stuff Documents 체인을 사용하여 완전한 RAG 파이프라인을 구현하세요.
- [x] 체인을 수동으로 구현해야 합니다.
- [x] 체인에 ConversationBufferMemory를 부여합니다.
- [x] 이 문서를 사용하여 RAG를 수행하세요: (https://gist.github.com/serranoarevalo/5acf755c2b8d83f1707ef266b82ea223)
- 다음과 같은 절차대로 구현하면 챌린지를 해결할 수 있습니다.
    2. [x] 문서 쪼개기 : CharacterTextSplitter 등 을 사용해서 문서를 작은 문서 조각들로 나눕니다.
    3. [x] 임베딩 생성 및 캐시 : OpenAIEmbeddings, CacheBackedEmbeddings 등 을 사용해 문서 조각들을 임베딩하고 임베딩을 저장합니다.
    4. [x] 벡터 스토어 생성 : FAISS 등 을 사용해서 임베딩된 문서들을 저장하고 검색할 수 있는 데이터베이스를 만듭니다.
    5. [x] 대화 메모리와 질문 처리 : ConversationBufferMemory를 사용해 대화 기록을 관리합니다.
    6. [x] 체인 연결 : 앞에서 구현한 컴포넌트들을 적절하게 체인으로 연결합니다.


In [ ]:
from langchain.chat_models.openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.vectorstores.faiss import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    temperature=0.1,
)
memory = ConversationBufferMemory(return_messages=True)

loader = UnstructuredFileLoader("./day4.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n\n",
    chunk_size=600,
    chunk_overlap=100,
)
file = loader.load_and_split(splitter)

cache_dir = LocalFileStore("./.day4_cache/")
embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(file, cached_embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Documents will be given. Answer the human question only with given context. If you don't know the answer, don't make it up:\n\n{context}"),
    ("human", "{question}"),
])

def load_memory(_):
    return memory.load_memory_variables({})["history"]

chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

def invoke_chain(question):
    result = chain.invoke(question)
    memory.save_context({"input":question}, {"output":result.content})

In [21]:
invoke_chain("Is Aaronson guilty?")
invoke_chain("What he wrote on the table?")
invoke_chain("Who is Julia?")

load_memory(_)

[HumanMessage(content='Is Aaronson guilty?'),
 AIMessage(content='The provided documents do not mention Aaronson or any details regarding his guilt or innocence.'),
 HumanMessage(content='What he wrote on the table?'),
 AIMessage(content='He wrote first in large clumsy capitals: \n\nFREEDOM IS SLAVERY\n\nThen almost without a pause he wrote beneath it:\n\nTWO AND TWO MAKE FIVE\n\nHe also wrote:\n\nGOD IS POWER'),
 HumanMessage(content='Who is Julia?'),
 AIMessage(content='Julia is a character that Winston loves deeply, and he experiences a strong emotional connection to her throughout the narrative. She is someone with whom he has had a romantic relationship, and he expresses a desire to protect her, even under extreme duress.'),
 HumanMessage(content='Is Aaronson guilty?'),
 AIMessage(content='The provided documents do not mention Aaronson or any details regarding his guilt. Therefore, I cannot answer the question.'),
 HumanMessage(content='What he wrote on the table?'),
 AIMessage(co